In [1]:
import cobra
from cobra.flux_analysis import flux_variability_analysis
from determine_trait_flux_basis import determine_trait_flux_basis
from enumerate_evolvex import enumerate_evolvex
from get_growth_supporting_niche12 import get_growth_supporting_niche12

In [19]:
model = cobra.io.read_sbml_model("iML1515.xml")

#1.Nullstiller objektivfunksjon og setter biomasse som objektiv
for rxn in model.reactions:
    rxn.objective_coefficient = 0
model.reactions.get_by_id("BIOMASS_Ec_iML1515_core_75p37M").objective_coefficient = 1
#BIOMASS_Ec_iJO1366_core_53p95M
#BIOMASS_Ec_iML1515_core_75p37M
#BIOMASS_SC5_notrace

In [20]:
#2.Create a copy for flux-basis determination
model_wt = model.copy()

#3.Run FVA to define wild-type flux ranges
fva_result = flux_variability_analysis(
    model_wt,
    reaction_list=model_wt.reactions,
    fraction_of_optimum=1.0
)
wtminmax = fva_result[['minimum', 'maximum']].to_numpy() #benyttes i determine_trait_flux_basis etterpå
#WT intervall er når modellen optimaliserer vekst. 
#lb og ub er for trait-modellen med ønsket objektivfunksjon. 

In [21]:
#4.Definerer den nye objektivfunksjonen
trait_model = model.copy() #M9 medium
for rxn in trait_model.reactions:
    rxn.objective_coefficient = 0

trait=trait_model.reactions.get_by_id("EX_succ_e")
trait.lower_bound=0 #blokkerer opptak. Hindre at den ikke øke fluxen ved å ta opp direkte fra medium. 
trait.upper_bound=1000 
trait.objective_coefficient = 1 #setter ønsket reaksjon som objektivfunksjon

#sjekker om det støtter vekst
sol = trait_model.optimize()
print(sol.objective_value)

17.137454545454542


In [22]:
#Basemedium, det universale som ligger i miljøet
lb_open = ["EX_ala__L_e","EX_glu__L_e","EX_gln__L_e","EX_asp__L_e","EX_ser__L_e","EX_gly_e","EX_leu__L_e","EX_lys__L_e", "EX_thm_e", "EX_btn_e"]

def open_uptake(trait_model, lb_open):
    for rid in lb_open:
        if rid in trait_model.reactions:
            trait_model.reactions.get_by_id(rid).lower_bound = -abs(10)

def aapen(trait_model, threshold=-1e-9):
    open_ex = []
    for r in trait_model.reactions:
        if r.id.startswith("EX_") and r.lower_bound < threshold:
            open_ex.append((r.id, r.lower_bound, r.upper_bound))
    return sorted(open_ex)

open_uptake(trait_model, lb_open)
open_uptakes = aapen(trait_model)
for x in open_uptakes[:100]:
    print(x)

('EX_ala__L_e', -10, 1000.0)
('EX_asp__L_e', -10, 1000.0)
('EX_btn_e', -10, 1000.0)
('EX_ca2_e', -1000.0, 1000.0)
('EX_cl_e', -1000.0, 1000.0)
('EX_co2_e', -1000.0, 1000.0)
('EX_cobalt2_e', -1000.0, 1000.0)
('EX_cu2_e', -1000.0, 1000.0)
('EX_fe2_e', -1000.0, 1000.0)
('EX_fe3_e', -1000.0, 1000.0)
('EX_glc__D_e', -10.0, 1000.0)
('EX_gln__L_e', -10, 1000.0)
('EX_glu__L_e', -10, 1000.0)
('EX_gly_e', -10, 1000.0)
('EX_h2o_e', -1000.0, 1000.0)
('EX_h_e', -1000.0, 1000.0)
('EX_k_e', -1000.0, 1000.0)
('EX_leu__L_e', -10, 1000.0)
('EX_lys__L_e', -10, 1000.0)
('EX_mg2_e', -1000.0, 1000.0)
('EX_mn2_e', -1000.0, 1000.0)
('EX_mobd_e', -1000.0, 1000.0)
('EX_na1_e', -1000.0, 1000.0)
('EX_nh4_e', -1000.0, 1000.0)
('EX_ni2_e', -1000.0, 1000.0)
('EX_o2_e', -1000.0, 1000.0)
('EX_pi_e', -1000.0, 1000.0)
('EX_sel_e', -1000.0, 1000.0)
('EX_ser__L_e', -10, 1000.0)
('EX_slnt_e', -1000.0, 1000.0)
('EX_so4_e', -1000.0, 1000.0)
('EX_thm_e', -10, 1000.0)
('EX_tungs_e', -1000.0, 1000.0)
('EX_zn2_e', -1000.0, 1000.

In [23]:
#6.Determine_trait_flux_basis
import numpy as np 
targets, dirs, signs, dirSolp, binRxns, fullSolOpt, solutionp = determine_trait_flux_basis(trait_model, 'max', wtminmax, 1, 1, 0.00001, 5)
#7.parse-funksjon som fikser evolvex input
targets_flat = targets[0]
dirs_flat = dirs[0]

evolveX_targets = [int(x) for x in targets_flat]
evolveX_target_dirs = list(dirs_flat)

# lag dict til evolveX_score
targets_dict = {model.reactions[i].id: d for i, d in zip(evolveX_targets, evolveX_target_dirs)}

print(targets_dict)



{'ICDHyr': 'DOWN', 'ASPTA': 'DOWN', 'PPS': 'UP', 'MDH': 'DOWN', 'FUM': 'DOWN', 'ASPT': 'UP', 'SERD_L': 'UP', 'ICL': 'UP', 'GLYCL': 'UP', 'PPC': 'UP', 'GHMT2r': 'DOWN', 'MALS': 'UP', 'GLUtex': 'UP', 'GLYtex': 'UP', 'CO2tpp': 'UP', 'GLNtex': 'UP', 'ASPtex': 'UP', 'SERtex': 'UP', 'SUCCtex': 'DOWN', 'NH4tpp': 'DOWN', 'GLUNpp': 'UP', 'NH4tex': 'DOWN', 'ALAtex': 'UP', 'H2Otpp': 'UP', 'CO2tex': 'UP', 'GLUDy': 'UP', 'PDH': 'UP', 'ALAt4pp': 'UP', 'GLYtpp': 'DOWN', 'SERt2rpp': 'UP'}


In [24]:
#8.Prepare EvolveX-like growth-constrained model
model_evolvex = model.copy()

#9.Stenger glukose og ammonium for å finne alternative ruter 
model_evolvex.reactions.get_by_id("EX_glc__D_e").lower_bound = 0
model_evolvex.reactions.get_by_id("EX_glc__D_e").upper_bound = 0

In [12]:
#12.lager kombinasjoner som støtter vekst med alternative kilder
condMap = {
    "carbon": ["EX_gal_e", "EX_fru_e", "EX_ac_e", "EX_pyr_e", "EX_succ_e", "EX_glyc_e", "EX_xyl__D_e"],
    "nitrogen": ["EX_gln__L_e", "EX_glu__L_e","EX_ala__L_e", "EX_ser__L_e", "EX_nh4_e"]
}

growth, uptakes, combos = get_growth_supporting_niche12(
    model_evolvex,
    condMap
)

print(growth, uptakes, combos)


1 EX_gal_e EX_gln__L_e EX_glu__L_e
2 EX_gal_e EX_gln__L_e EX_ala__L_e
3 EX_gal_e EX_gln__L_e EX_ser__L_e
4 EX_gal_e EX_gln__L_e EX_nh4_e
5 EX_gal_e EX_glu__L_e EX_ala__L_e
6 EX_gal_e EX_glu__L_e EX_ser__L_e
7 EX_gal_e EX_glu__L_e EX_nh4_e
8 EX_gal_e EX_ala__L_e EX_ser__L_e
9 EX_gal_e EX_ala__L_e EX_nh4_e
10 EX_gal_e EX_ser__L_e EX_nh4_e
11 EX_fru_e EX_gln__L_e EX_glu__L_e
12 EX_fru_e EX_gln__L_e EX_ala__L_e
13 EX_fru_e EX_gln__L_e EX_ser__L_e
14 EX_fru_e EX_gln__L_e EX_nh4_e
15 EX_fru_e EX_glu__L_e EX_ala__L_e
16 EX_fru_e EX_glu__L_e EX_ser__L_e
17 EX_fru_e EX_glu__L_e EX_nh4_e
18 EX_fru_e EX_ala__L_e EX_ser__L_e
19 EX_fru_e EX_ala__L_e EX_nh4_e
20 EX_fru_e EX_ser__L_e EX_nh4_e
21 EX_ac_e EX_gln__L_e EX_glu__L_e
22 EX_ac_e EX_gln__L_e EX_ala__L_e
23 EX_ac_e EX_gln__L_e EX_ser__L_e
24 EX_ac_e EX_gln__L_e EX_nh4_e


/opt/anaconda3/envs/cplex_env/lib/python3.10/site-packages/cobra/util/solver.py:554: UserWarning: Solver status is 'infeasible'.
  warn(f"Solver status is '{status}'.", UserWarning)


25 EX_ac_e EX_glu__L_e EX_ala__L_e
26 EX_ac_e EX_glu__L_e EX_ser__L_e
27 EX_ac_e EX_glu__L_e EX_nh4_e
28 EX_ac_e EX_ala__L_e EX_ser__L_e
29 EX_ac_e EX_ala__L_e EX_nh4_e
30 EX_ac_e EX_ser__L_e EX_nh4_e
31 EX_pyr_e EX_gln__L_e EX_glu__L_e
32 EX_pyr_e EX_gln__L_e EX_ala__L_e
33 EX_pyr_e EX_gln__L_e EX_ser__L_e
34 EX_pyr_e EX_gln__L_e EX_nh4_e
35 EX_pyr_e EX_glu__L_e EX_ala__L_e
36 EX_pyr_e EX_glu__L_e EX_ser__L_e
37 EX_pyr_e EX_glu__L_e EX_nh4_e
38 EX_pyr_e EX_ala__L_e EX_ser__L_e
39 EX_pyr_e EX_ala__L_e EX_nh4_e
40 EX_pyr_e EX_ser__L_e EX_nh4_e
41 EX_succ_e EX_gln__L_e EX_glu__L_e
42 EX_succ_e EX_gln__L_e EX_ala__L_e
43 EX_succ_e EX_gln__L_e EX_ser__L_e
44 EX_succ_e EX_gln__L_e EX_nh4_e
45 EX_succ_e EX_glu__L_e EX_ala__L_e
46 EX_succ_e EX_glu__L_e EX_ser__L_e
47 EX_succ_e EX_glu__L_e EX_nh4_e
48 EX_succ_e EX_ala__L_e EX_ser__L_e
49 EX_succ_e EX_ala__L_e EX_nh4_e
50 EX_succ_e EX_ser__L_e EX_nh4_e
51 EX_glyc_e EX_gln__L_e EX_glu__L_e
52 EX_glyc_e EX_gln__L_e EX_ala__L_e
53 EX_glyc_e EX_gln

In [25]:
combos=[('EX_gal_e', 'EX_gln__L_e', 'EX_glu__L_e'), ('EX_gal_e', 'EX_gln__L_e', 'EX_ala__L_e'), ('EX_gal_e', 'EX_gln__L_e', 'EX_ser__L_e'), ('EX_gal_e', 'EX_gln__L_e', 'EX_nh4_e'), ('EX_gal_e', 'EX_glu__L_e', 'EX_ala__L_e'), ('EX_gal_e', 'EX_glu__L_e', 'EX_ser__L_e'), ('EX_gal_e', 'EX_glu__L_e', 'EX_nh4_e'), ('EX_gal_e', 'EX_ala__L_e', 'EX_ser__L_e'), ('EX_gal_e', 'EX_ala__L_e', 'EX_nh4_e'), ('EX_gal_e', 'EX_ser__L_e', 'EX_nh4_e'), ('EX_fru_e', 'EX_gln__L_e', 'EX_glu__L_e'), ('EX_fru_e', 'EX_gln__L_e', 'EX_ala__L_e'), ('EX_fru_e', 'EX_gln__L_e', 'EX_ser__L_e'), ('EX_fru_e', 'EX_gln__L_e', 'EX_nh4_e'), ('EX_fru_e', 'EX_glu__L_e', 'EX_ala__L_e'), ('EX_fru_e', 'EX_glu__L_e', 'EX_ser__L_e'), ('EX_fru_e', 'EX_glu__L_e', 'EX_nh4_e'), ('EX_fru_e', 'EX_ala__L_e', 'EX_ser__L_e'), ('EX_fru_e', 'EX_ala__L_e', 'EX_nh4_e'), ('EX_fru_e', 'EX_ser__L_e', 'EX_nh4_e'), ('EX_ac_e', 'EX_gln__L_e', 'EX_glu__L_e'), ('EX_ac_e', 'EX_gln__L_e', 'EX_ala__L_e'), ('EX_ac_e', 'EX_gln__L_e', 'EX_ser__L_e'), ('EX_ac_e', 'EX_gln__L_e', 'EX_nh4_e'), ('EX_ac_e', 'EX_glu__L_e', 'EX_ala__L_e'), ('EX_ac_e', 'EX_glu__L_e', 'EX_ser__L_e'), ('EX_ac_e', 'EX_glu__L_e', 'EX_nh4_e'), ('EX_ac_e', 'EX_ala__L_e', 'EX_ser__L_e'), ('EX_ac_e', 'EX_ala__L_e', 'EX_nh4_e'), ('EX_ac_e', 'EX_ser__L_e', 'EX_nh4_e'), ('EX_pyr_e', 'EX_gln__L_e', 'EX_glu__L_e'), ('EX_pyr_e', 'EX_gln__L_e', 'EX_ala__L_e'), ('EX_pyr_e', 'EX_gln__L_e', 'EX_ser__L_e'), ('EX_pyr_e', 'EX_gln__L_e', 'EX_nh4_e'), ('EX_pyr_e', 'EX_glu__L_e', 'EX_ala__L_e'), ('EX_pyr_e', 'EX_glu__L_e', 'EX_ser__L_e'), ('EX_pyr_e', 'EX_glu__L_e', 'EX_nh4_e'), ('EX_pyr_e', 'EX_ala__L_e', 'EX_ser__L_e'), ('EX_pyr_e', 'EX_ala__L_e', 'EX_nh4_e'), ('EX_pyr_e', 'EX_ser__L_e', 'EX_nh4_e'), ('EX_succ_e', 'EX_gln__L_e', 'EX_glu__L_e'), ('EX_succ_e', 'EX_gln__L_e', 'EX_ala__L_e'), ('EX_succ_e', 'EX_gln__L_e', 'EX_ser__L_e'), ('EX_succ_e', 'EX_gln__L_e', 'EX_nh4_e'), ('EX_succ_e', 'EX_glu__L_e', 'EX_ala__L_e'), ('EX_succ_e', 'EX_glu__L_e', 'EX_ser__L_e'), ('EX_succ_e', 'EX_glu__L_e', 'EX_nh4_e'), ('EX_succ_e', 'EX_ala__L_e', 'EX_ser__L_e'), ('EX_succ_e', 'EX_ala__L_e', 'EX_nh4_e'), ('EX_succ_e', 'EX_ser__L_e', 'EX_nh4_e'), ('EX_glyc_e', 'EX_gln__L_e', 'EX_glu__L_e'), ('EX_glyc_e', 'EX_gln__L_e', 'EX_ala__L_e'), ('EX_glyc_e', 'EX_gln__L_e', 'EX_ser__L_e'), ('EX_glyc_e', 'EX_gln__L_e', 'EX_nh4_e'), ('EX_glyc_e', 'EX_glu__L_e', 'EX_ala__L_e'), ('EX_glyc_e', 'EX_glu__L_e', 'EX_ser__L_e'), ('EX_glyc_e', 'EX_glu__L_e', 'EX_nh4_e'), ('EX_glyc_e', 'EX_ala__L_e', 'EX_ser__L_e'), ('EX_glyc_e', 'EX_ala__L_e', 'EX_nh4_e'), ('EX_glyc_e', 'EX_ser__L_e', 'EX_nh4_e'), ('EX_xyl__D_e', 'EX_gln__L_e', 'EX_glu__L_e'),
('EX_xyl__D_e', 'EX_gln__L_e', 'EX_ala__L_e'),
('EX_xyl__D_e', 'EX_gln__L_e', 'EX_ser__L_e'),
('EX_xyl__D_e', 'EX_gln__L_e', 'EX_nh4_e'),
('EX_xyl__D_e', 'EX_glu__L_e', 'EX_ala__L_e'),
('EX_xyl__D_e', 'EX_glu__L_e', 'EX_ser__L_e'),
('EX_xyl__D_e', 'EX_glu__L_e', 'EX_nh4_e'),
('EX_xyl__D_e', 'EX_ala__L_e', 'EX_ser__L_e'),
('EX_xyl__D_e', 'EX_ala__L_e', 'EX_nh4_e'),
('EX_xyl__D_e', 'EX_ser__L_e', 'EX_nh4_e')]


In [11]:
#Basemedium, det universale som ligger i miljøet
def aapen(model_evolvex, threshold=-1e-9):
    open_ex = []
    for r in model_evolvex.reactions:
        if r.id.startswith("EX_") and r.lower_bound < threshold:
            open_ex.append((r.id, r.lower_bound, r.upper_bound))
    return sorted(open_ex)

open_uptakes = aapen(model_evolvex)
for x in open_uptakes[:100]:
    print(x)

('EX_ca2_e', -10000.0, 10000.0)
('EX_cl_e', -10000.0, 10000.0)
('EX_co2_e', -10000.0, 10000.0)
('EX_cobalt2_e', -10000.0, 10000.0)
('EX_cu2_e', -10000.0, 10000.0)
('EX_fe2_e', -10000.0, 10000.0)
('EX_fe3_e', -10000.0, 10000.0)
('EX_h2o_e', -10000.0, 10000.0)
('EX_h_e', -10000.0, 10000.0)
('EX_k_e', -10000.0, 10000.0)
('EX_mg2_e', -10000.0, 10000.0)
('EX_mn2_e', -10000.0, 10000.0)
('EX_mobd_e', -10000.0, 10000.0)
('EX_na1_e', -10000.0, 10000.0)
('EX_ni2_e', -10000.0, 10000.0)
('EX_o2_e', -10000.0, 10000.0)
('EX_pi_e', -10000.0, 10000.0)
('EX_sel_e', -10000.0, 10000.0)
('EX_slnt_e', -10000.0, 10000.0)
('EX_so4_e', -10000.0, 10000.0)
('EX_tungs_e', -10000.0, 10000.0)
('EX_zn2_e', -10000.0, 10000.0)


In [26]:
from EVOLVEX_SCORE import setBounds, createSecretion
#13.lager envs fra get_growth_supporting_niche1 og finner optimality
envs = []
for (c, n1, n2) in combos:   
    env = {
        c: "comp",
        n1: "comp",
        n2: "comp",
        "EX_glc__D_e": "inh",
        #"EX_lac__D_e": "inh"
    }
    envs.append(env)

components = sorted(set(rxn for combo in combos for rxn in combo))
setBounds(model_evolvex)
for ex in components:
    createSecretion(model_evolvex, ex)

for ex in components:
    r = model_evolvex.reactions.get_by_id(ex)
    r.lower_bound = 0
    r.upper_bound = 0

In [27]:
#14.sammenligner targets med alle mulige envs for å gi en evolvex score på beste miljø
strength, comp = enumerate_evolvex(model_evolvex, envs, targets_dict)

print(strength)

/opt/anaconda3/envs/cplex_env/lib/python3.10/site-packages/cplex/_internal/_pycplex.py:1632: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  return _pycplex_platform.Pylolmat_to_CHBmat(lolmat, env_lp_ptr, py_row_col)


1/70 strength=28.3741935026611 comp=['EX_gal_e', 'EX_gln__L_e', 'EX_glu__L_e']


/opt/anaconda3/envs/cplex_env/lib/python3.10/site-packages/cplex/_internal/_pycplex.py:1632: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  return _pycplex_platform.Pylolmat_to_CHBmat(lolmat, env_lp_ptr, py_row_col)


2/70 strength=28.341935406879408 comp=['EX_gal_e', 'EX_gln__L_e', 'EX_ala__L_e']


/opt/anaconda3/envs/cplex_env/lib/python3.10/site-packages/cplex/_internal/_pycplex.py:1632: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  return _pycplex_platform.Pylolmat_to_CHBmat(lolmat, env_lp_ptr, py_row_col)


...

68/70 strength=31.267241366380837 comp=['EX_xyl__D_e', 'EX_ala__L_e', 'EX_ser__L_e']


/opt/anaconda3/envs/cplex_env/lib/python3.10/site-packages/cplex/_internal/_pycplex.py:1632: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  return _pycplex_platform.Pylolmat_to_CHBmat(lolmat, env_lp_ptr, py_row_col)


69/70 strength=31.92241371242412 comp=['EX_xyl__D_e', 'EX_ala__L_e', 'EX_nh4_e']


/opt/anaconda3/envs/cplex_env/lib/python3.10/site-packages/cplex/_internal/_pycplex.py:1632: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  return _pycplex_platform.Pylolmat_to_CHBmat(lolmat, env_lp_ptr, py_row_col)


70/70 strength=34.82758616068715 comp=['EX_xyl__D_e', 'EX_ser__L_e', 'EX_nh4_e']
[28.3741935026611, 28.341935406879408, 31.245161262424773, 31.890322460718465, 28.341935435661153, 31.245161263909903, 31.890322466892577, 31.21290317633766, 31.858064397333504, 34.76129022751739, 28.178723211197248, 28.114893437331116, 31.008510534018495, 31.604255116711567, 28.114893505415573, 31.008510537904407, 31.604255114096564, 30.944680702337063, 31.540425455167433, 34.434042515294216, 20.893939364750814, 19.636363563733486, 22.166666598658498, 20.893939362240097, 19.63636352523472, 22.16666660089149, 20.8939393723358, 15.92558137166357, 13.599999979379877, 29.711111090642973, 23.060605931838488, 21.803030237470058, 24.333333272770417, 23.060605931344284, 21.80303023511503, 24.33333330436286, 23.060605930966453, 18.413953402059004, 16.08837203469983, 31.488888818397772, 22.075757492905485, 20.818181753332794, 23.348484811398674, 22.075757453301843, 20.818181751426962, 23.348484812844934, 22.0757574